In [ ]:
!pip install pettingzoo[classic] pygame matplotlib numpy
print("Установка завершена. Перезапустите среду выполнения (Runtime -> Restart runtime)!")

import numpy as np
from pettingzoo.classic import tictactoe_v3
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import random

plt.rcParams['animation.embed_limit'] = 100

def check_win(board):
    """Проверяет победу для текущего состояния доски"""
    # Проверяем строки
    for i in range(3):
        if np.all(board[i,:,0] == 1): return 'player_1'  # X побеждает
        if np.all(board[i,:,1] == 1): return 'player_2'  # O побеждает

    # Проверяем столбцы
    for j in range(3):
        if np.all(board[:,j,0] == 1): return 'player_1'
        if np.all(board[:,j,1] == 1): return 'player_2'

    # Проверяем диагонали
    if np.all(np.diag(board[:,:,0]) == 1): return 'player_1'
    if np.all(np.diag(board[:,:,1]) == 1): return 'player_2'
    if np.all(np.diag(np.fliplr(board[:,:,0])) == 1): return 'player_1'
    if np.all(np.diag(np.fliplr(board[:,:,1])) == 1): return 'player_2'

    return None  # Если победителя нет

# Создаем среду для игры
env = tictactoe_v3.env()
env.reset()  # Сбрасываем среду в начальное состояние

# Сохраняем полную историю игры
history = []
current_board = np.zeros((3, 3, 2))  # Начинаем с пустой доски (3x3 для двух игроков)

# Основной игровой цикл
for agent in env.agent_iter():  # По очереди для каждого игрока
    observation, reward, done, truncation, info = env.last()  # Получаем последнее состояние

    # Обновляем доску на основе текущего наблюдения
    current_board = observation["observation"].copy()
    winner = check_win(current_board)  # Проверяем, есть ли победитель

    # Сохраняем полное состояние игры в историю
    history.append({
        'board': current_board.copy(),  # Текущее состояние доски
        'agent': agent,  # Текущий игрок
        'action': info.get('action') if 'action' in info else None,  # Последнее действие
        'winner': winner,  # Победитель (если есть)
        'done': done or (winner is not None)  # Флаг завершения игры
    })

    # Завершаем игру, если есть победитель или игра окончена
    if done or (winner is not None):
        break

    # Выбираем случайный допустимый ход
    mask = observation["action_mask"]  # Маска допустимых ходов
    legal_moves = np.where(mask == 1)[0]  # Индексы допустимых ходов
    if len(legal_moves) > 0:
        action = random.choice(legal_moves)  # Случайный выбор хода
        env.step(action)  # Совершаем ход в среде
    else:
        break  # Если нет допустимых ходов

env.close()  # Закрываем среду после игры

# Создаем анимацию игры
fig, ax = plt.subplots(figsize=(6, 6))

def update(frame_num):
    """Функция обновления кадра анимации"""
    ax.clear()  # Очищаем оси
    ax.set_xticks([])  # Убираем метки на осях
    ax.set_yticks([])

    frame = history[frame_num]  # Получаем текущий кадр из истории
    board = frame['board']  # Получаем состояние доски

    # Рисуем игровую сетку
    for i in range(1, 3):
        ax.axhline(i - 0.5, color='black', linewidth=2)
        ax.axvline(i - 0.5, color='black', linewidth=2)

    # Рисуем все сделанные ходы на текущий момент
    for y in range(3):
        for x in range(3):
            if board[y][x][0] == 1:  # Крестик (игрок 1)
                # Рисуем две диагональные линии для крестика
                ax.plot([x-0.4, x+0.4], [2-y+0.4, 2-y-0.4], color='red', linewidth=4)
                ax.plot([x-0.4, x+0.4], [2-y-0.4, 2-y+0.4], color='red', linewidth=4)
            elif board[y][x][1] == 1:  # Нолик (игрок 2)
                # Рисуем круг для нолика
                circle = plt.Circle((x, 2-y), 0.3, color='blue', fill=False, linewidth=4)
                ax.add_patch(circle)

    # Подсвечиваем последний сделанный ход
    if frame['action'] is not None:
        y, x = frame['action'] // 3, frame['action'] % 3  # Преобразуем индекс в координаты
        rect = plt.Rectangle((x-0.5, 2-y-0.5), 1, 1,
                           fill=False, edgecolor='green', linestyle='--', linewidth=2)
        ax.add_patch(rect)

    # Устанавливаем границы поля
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(-0.5, 2.5)

    # Отображаем информацию о ходе игры
    if frame['done'] or frame['winner']:
        if frame['winner']:
            # Определяем победителя
            winner = 'Крестики (X)' if frame['winner'] == 'player_1' else 'Нолики (O)'
            ax.set_title(f"Победа {winner}!", fontsize=14, color='green', pad=20)
        else:
            ax.set_title("Ничья!", fontsize=14, color='blue', pad=20)
    else:
        # Отображаем текущего игрока
        current_player = 'Крестики (X)' if frame['agent'] == 'player_1' else 'Нолики (O)'
        ax.set_title(f"Ход {frame_num+1}: {current_player}", fontsize=12, pad=20)

anim = animation.FuncAnimation(
    fig, update, frames=len(history), interval=800, repeat=False
)

plt.close()
HTML(anim.to_html5_video())

Установка завершена. Перезапустите среду выполнения (Runtime -> Restart runtime)!
